In [ ]:
import apoc
import napari
import numpy as np
from pathlib import Path
from skimage.io import imread
from tifffile import imwrite
from utils import list_images, read_image

In [ ]:
image_folder = "./training_data/mtb_train_data/images"
annotation_folder = "./training_data/mtb_train_data/annotations"
grayscale_annotation_folder = "./training_data/mtb_train_data/grayscale_annotations"

In [ ]:
# Convert RGB annotations to 2D grayscale TIFFs so shapes match the images
# Labels live in one RGB channel (green here); take max across channels
ann_dir = Path(annotation_folder)
out_dir = Path(grayscale_annotation_folder)
out_dir.mkdir(parents=True, exist_ok=True)

for ann_path in sorted(ann_dir.iterdir()):
    if not ann_path.is_file():
        continue

    ann = imread(ann_path)

    # Drop channel axis from RGB/RGBA (H, W, C) or (C, H, W)
    if ann.ndim == 3:
        if ann.shape[-1] in (3, 4):
            ann = ann[..., :3].max(axis=-1)
        elif ann.shape[0] in (3, 4):
            ann = ann[:3].max(axis=0)
        else:
            raise ValueError(f"Unexpected annotation shape {ann.shape} in {ann_path.name}")

    out_path = out_dir / f"{ann_path.stem}.tif"
    imwrite(out_path, ann)
    print(f"{ann_path.name}: unique labels {sorted(set(ann.flat))} nonzero={int((ann > 0).sum())} -> {out_path.name}")

print(f"Saved grayscale annotations to {out_dir}")

In [ ]:
# Setup classifer and where it should be saved
cl_filename = "./models/Mtb_segmenter.cl"
apoc.erase_classifier(cl_filename)
segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)

# Setup feature set used for training
features = apoc.PredefinedFeatureSet.object_size_1_to_5_px.value #TODO: Check

# Train classifier on folders
apoc.train_classifier_from_image_folders(
    segmenter, 
    features, 
    image = image_folder, # Grayscale processed images, no RGB
    ground_truth = grayscale_annotation_folder)

In [ ]:
# Load the trained segmenter
viewer = napari.Viewer(ndisplay=2)
segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)

In [ ]:
# Copy the path where your images are stored, you can use absolute or relative paths to point at other disk locations
directory_path = Path(r"X:\Shirin\260720_SK0074_iPSDMsWT-NINJ1KO_MtbInf-MOI2-20_NoOpsNoSon\260721_SK_SK0074_Exp01_ConfocalMic\RawData")

# Iterate through the .czi and .nd2 files in the directory
images = list_images(directory_path, format="nd2")

# Image size reduction (downsampling) to improve processing times (slicing, not lossless compression)
slicing_factor_xy = None # Use 2 or 4 for downsampling in xy (None for lossless)

images

In [ ]:
# Test classifier in a few images 
image = images[0]

# Read image, apply slicing if needed and return filename and img as a np array
img, filename = read_image(image, slicing_factor_xy)

# Perform MIP before feeding the image into the pipeline
img = np.max(img, axis=0)

viewer = napari.Viewer(ndisplay=2)

predicted_labels = segmenter.predict(img[1])
viewer.add_image(img[1])
viewer.add_labels(predicted_labels)